# Gallery-conditioned CLIP on RSTPReid (Kaggle)

Enable GPU (T4 x2), attach `hoanggv/tbps-benchmark`, then run cells top to bottom. The recipe matches the CLIP TBPS / IRRA reference settings for RSTPReid: OpenAI ViT-B/16, 384×128 images, 77-token CLIP text input, 64 image-caption pairs per identity-balanced batch, Adam at 1e-5, SDM + ID losses, and a 60-epoch cosine schedule with 5 warmup epochs. Gallery self-attention is the added module.

In [ ]:
# Attach the Kaggle dataset in the notebook's Input panel first.
from pathlib import Path
roots = list(Path('/kaggle/input').glob('*'))
[(str(p), [x.name for x in p.iterdir()][:10]) for p in roots]

## Get the public project

Once the repository is published, set `REPO_URL` to its HTTPS URL. This fetches the training and evaluation code into the Kaggle session.

In [ ]:
REPO_URL = 'https://github.com/lqb464/gallery-conditioned-tbps.git'
!git clone -q $REPO_URL /kaggle/working/gallery-conditioned-tbps
%cd /kaggle/working/gallery-conditioned-tbps
!pip -q install -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), 'In Kaggle: Settings → Accelerator → GPU'
print(torch.cuda.get_device_name(0), '| GPUs:', torch.cuda.device_count())
DATA_ROOT = '/kaggle/input/tbps-benchmark'  # change to the mounted directory shown above
OUT = '/kaggle/working/gallery_tbps'
from pathlib import Path
assert Path(DATA_ROOT).exists(), f'Dataset directory not found: {DATA_ROOT}'

In [ ]:
# Essential forward + gradient smoke check (two-image gallery).
import open_clip
from tbps.model import GalleryCLIP
clip, _, _ = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai', force_image_size=(384, 128))
smoke = GalleryCLIP(clip).cuda().eval()
features = torch.nn.functional.normalize(torch.randn(1, 3, 512, device='cuda'), dim=-1)
context_a = smoke.contextualize(features[:, :2])
context_b = smoke.contextualize(features[:, [0, 2]])
assert not torch.allclose(context_a[:, 0], context_b[:, 0]), 'Gallery composition did not affect context'
smoke.train()
tokens = open_clip.get_tokenizer('ViT-B-16')(['a person in a dark coat', 'a person wearing a light shirt']).cuda()
scores = smoke(torch.randn(1, 2, 3, 384, 128, device='cuda'), tokens)
assert scores.shape == (2, 2), scores.shape
scores.sum().backward()
assert smoke.gallery.encoder.layers[0].self_attn.in_proj_weight.grad is not None
print('forward shape:', tuple(scores.shape), '| gallery gradient: OK | composition sensitivity: OK')
del smoke, clip, scores, tokens, features, context_a, context_b
torch.cuda.empty_cache()

## Inspect annotation and image layout

The loader supports JSON annotations and parquet exports. If this cell lists no JSON/parquet files, inspect the printed tree and adjust `DATA_ROOT` to the directory containing the annotation file and images.

In [ ]:
root = Path(DATA_ROOT)
print('JSON:', [str(p.relative_to(root)) for p in root.rglob('*.json')][:20])
print('Parquet:', [str(p.relative_to(root)) for p in root.rglob('*.parquet')][:20])
print('Images:', len(list(root.rglob('*.jpg'))), len(list(root.rglob('*.png'))))
from tbps.data import load_records
for split in ('train', 'val', 'test'):
    try: print(split, len(load_records(DATA_ROOT, split)))
    except RuntimeError as e: print(split, e)

## Train

This fine-tunes the entire OpenAI CLIP ViT-B/16 and the 2-layer gallery Transformer. The reference config uses 16 identities × 4 image-caption pairs (batch 64). For T4 16 GB, this notebook overrides it to batch 32 (8 identities × 4) with accumulation 2, preserving effective optimizer batch 64 while keeping each attention group intact. One GPU processes each complete group; the second T4 remains idle because this simple runner does not shard full-gallery attention.

In [ ]:
!python -m tbps.train --config configs/clip_tbps_rstpreid.yaml --data-root "$DATA_ROOT" --output-dir "$OUT" --batch-size 32 --accumulation-steps 2 --workers 2

## Evaluate test split

The evaluator encodes/caches the CLIP gallery features for this run, contextualizes the complete test gallery, then reports text-to-image and image-to-text Recall@1/5/10. Re-run contextualization whenever the gallery composition changes.

In [ ]:
!python -m tbps.evaluate --data-root "$DATA_ROOT" --checkpoint "$OUT/best.pt" --split test --batch-size 64

In [ ]:
# Keep the checkpoint as a Kaggle output artifact.
from pathlib import Path
[(p.name, round(p.stat().st_size / 1024**2, 1), 'MB') for p in Path(OUT).glob('*.pt')]